<a href="https://colab.research.google.com/github/dsatish1252/sqlite-python-implementation/blob/main/02_SQL_Deep_Dive_Joins_Advanced_SQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python + RDBMS + SQL Training  
## Notebook 2 of 3: Deep Dive into SQL Queries, Joins and Advanced SQL

### Duration
This notebook is designed for approximately **4 hours** of hands-on learning.

### What you will learn
- Filtering with advanced conditions
- Joins: inner, left, cross, self
- Aggregation: count, sum, average, min, max
- `GROUP BY` and `HAVING`
- Subqueries
- Common Table Expressions
- CASE statements
- Window functions
- Views, indexes and transactions


## 1. Why Advanced SQL Matters

Business questions usually need more than a simple `SELECT`.

Examples:
- Which course generated highest revenue?
- Which student has pending payment?
- Which city has the highest number of students?
- Which course has more than one enrollment?
- Which department generated highest paid revenue?
- Which instructor is generating maximum business?

These questions need joins, grouping, conditional logic and analytical SQL.


## 2. Setup Database


In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

def connect(db_name):
    conn = sqlite3.connect(db_name)
    conn.execute("PRAGMA foreign_keys = ON;")
    return conn

def exec_script(conn, script):
    conn.executescript(script)
    conn.commit()
    print("Script executed successfully.")

def execute(conn, sql, params=None):
    if params is None:
        params = []
    cur = conn.execute(sql, params)
    conn.commit()
    print(f"Rows affected: {cur.rowcount}")
    return cur

def query(conn, sql, params=None):
    if params is None:
        params = []
    return pd.read_sql_query(sql, conn, params=params)

def tables(conn):
    return query(conn, """
    SELECT name
    FROM sqlite_master
    WHERE type='table' AND name NOT LIKE 'sqlite_%'
    ORDER BY name;
    """)

def show_schema(conn, table_name):
    print(f"\nSchema: {table_name}")
    display(query(conn, f"PRAGMA table_info({table_name});"))
    fk = query(conn, f"PRAGMA foreign_key_list({table_name});")
    if not fk.empty:
        print("Foreign Keys:")
        display(fk)

DB_NAME = "rdbms_notebook_02.db"
path = Path(DB_NAME)
if path.exists():
    path.unlink()

conn = connect(DB_NAME)
print("Connected to:", DB_NAME)


Connected to: rdbms_notebook_02.db


In [ ]:
exec_script(conn, """
CREATE TABLE departments (
    department_id INTEGER PRIMARY KEY,
    department_name TEXT NOT NULL UNIQUE
);

CREATE TABLE students (
    student_id INTEGER PRIMARY KEY,
    student_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    city TEXT,
    registration_date DATE NOT NULL
);

CREATE TABLE instructors (
    instructor_id INTEGER PRIMARY KEY,
    instructor_name TEXT NOT NULL,
    email TEXT NOT NULL UNIQUE,
    department_id INTEGER NOT NULL,
    FOREIGN KEY (department_id) REFERENCES departments(department_id)
);

CREATE TABLE courses (
    course_id INTEGER PRIMARY KEY,
    course_name TEXT NOT NULL,
    department_id INTEGER NOT NULL,
    instructor_id INTEGER NOT NULL,
    fee REAL NOT NULL CHECK (fee >= 0),
    level TEXT NOT NULL CHECK (level IN ('Beginner', 'Intermediate', 'Advanced')),
    FOREIGN KEY (department_id) REFERENCES departments(department_id),
    FOREIGN KEY (instructor_id) REFERENCES instructors(instructor_id)
);

CREATE TABLE enrollments (
    enrollment_id INTEGER PRIMARY KEY,
    student_id INTEGER NOT NULL,
    course_id INTEGER NOT NULL,
    enrollment_date DATE NOT NULL,
    status TEXT NOT NULL DEFAULT 'Active' CHECK (status IN ('Active', 'Completed', 'Cancelled')),
    UNIQUE(student_id, course_id),
    FOREIGN KEY (student_id) REFERENCES students(student_id),
    FOREIGN KEY (course_id) REFERENCES courses(course_id)
);

CREATE TABLE payments (
    payment_id INTEGER PRIMARY KEY,
    enrollment_id INTEGER NOT NULL UNIQUE,
    amount REAL NOT NULL CHECK (amount >= 0),
    payment_date DATE NOT NULL,
    payment_status TEXT NOT NULL CHECK (payment_status IN ('Paid', 'Pending', 'Refunded')),
    FOREIGN KEY (enrollment_id) REFERENCES enrollments(enrollment_id)
);
""")
exec_script(conn, """
INSERT INTO departments VALUES
(1, 'Data Science'),
(2, 'Software Engineering'),
(3, 'Business Analytics');

INSERT INTO students VALUES
(1, 'Rahul Kumar', 'rahul.kumar@example.com', 'Patna', '2026-01-05'),
(2, 'Priya Singh', 'priya.singh@example.com', 'Kolkata', '2026-01-06'),
(3, 'Amit Raj', 'amit.raj@example.com', 'Delhi', '2026-01-07'),
(4, 'Sneha Verma', 'sneha.verma@example.com', 'Patna', '2026-01-10'),
(5, 'Aditya Sharma', 'aditya.sharma@example.com', 'Mumbai', '2026-01-12'),
(6, 'Neha Gupta', 'neha.gupta@example.com', 'Ranchi', '2026-01-13');

INSERT INTO instructors VALUES
(1, 'Dr. Meera Iyer', 'meera.iyer@example.com', 1),
(2, 'Arjun Sen', 'arjun.sen@example.com', 2),
(3, 'Kavita Rao', 'kavita.rao@example.com', 3);

INSERT INTO courses VALUES
(101, 'Python for Beginners', 2, 2, 4999, 'Beginner'),
(102, 'SQL and RDBMS Masterclass', 1, 1, 6999, 'Beginner'),
(103, 'Machine Learning Basics', 1, 1, 11999, 'Intermediate'),
(104, 'Business Dashboarding', 3, 3, 8999, 'Intermediate'),
(105, 'Advanced Data Engineering', 1, 1, 15999, 'Advanced');

INSERT INTO enrollments VALUES
(1001, 1, 101, '2026-02-01', 'Active'),
(1002, 1, 102, '2026-02-03', 'Completed'),
(1003, 2, 102, '2026-02-04', 'Active'),
(1004, 3, 103, '2026-02-05', 'Active'),
(1005, 4, 104, '2026-02-07', 'Cancelled'),
(1006, 5, 105, '2026-02-08', 'Active'),
(1007, 6, 101, '2026-02-08', 'Completed'),
(1008, 2, 104, '2026-02-09', 'Active');

INSERT INTO payments VALUES
(501, 1001, 4999, '2026-02-01', 'Paid'),
(502, 1002, 6999, '2026-02-03', 'Paid'),
(503, 1003, 6999, '2026-02-04', 'Pending'),
(504, 1004, 11999, '2026-02-05', 'Paid'),
(505, 1005, 0, '2026-02-07', 'Refunded'),
(506, 1006, 15999, '2026-02-08', 'Paid'),
(507, 1007, 4999, '2026-02-08', 'Paid'),
(508, 1008, 8999, '2026-02-09', 'Pending');
""")
exec_script(conn, """
INSERT INTO departments VALUES
(4, 'Cloud Computing');

INSERT INTO students VALUES
(7, 'Rohan Das', 'rohan.das@example.com', 'Pune', '2026-01-15'),
(8, 'Farhan Ali', 'farhan.ali@example.com', 'Patna', '2026-01-18'),
(9, 'Mansi Roy', 'mansi.roy@example.com', 'Kolkata', '2026-01-20'),
(10, 'Kiran Nair', 'kiran.nair@example.com', 'Bengaluru', '2026-01-22');

INSERT INTO instructors VALUES
(4, 'Ravi Menon', 'ravi.menon@example.com', 4);

INSERT INTO courses VALUES
(106, 'Cloud Database Fundamentals', 4, 4, 10999, 'Beginner'),
(107, 'Data Visualization with Python', 3, 3, 7999, 'Beginner'),
(108, 'MLOps Fundamentals', 1, 1, 13999, 'Advanced');

INSERT INTO enrollments VALUES
(1009, 7, 106, '2026-02-10', 'Active'),
(1010, 8, 107, '2026-02-11', 'Completed'),
(1011, 9, 102, '2026-02-12', 'Active'),
(1012, 10, 108, '2026-02-13', 'Active'),
(1013, 8, 101, '2026-02-14', 'Active'),
(1014, 3, 102, '2026-02-15', 'Completed');

INSERT INTO payments VALUES
(509, 1009, 10999, '2026-02-10', 'Paid'),
(510, 1010, 7999, '2026-02-11', 'Paid'),
(511, 1011, 6999, '2026-02-12', 'Pending'),
(512, 1012, 13999, '2026-02-13', 'Paid'),
(513, 1013, 4999, '2026-02-14', 'Paid'),
(514, 1014, 6999, '2026-02-15', 'Paid');
""")
tables(conn)


Script executed successfully.
Script executed successfully.
Script executed successfully.


,name
0,courses
1,departments
2,enrollments
3,instructors
4,payments
5,students


## 3. SQL Logical Processing Order

SQL is written like this:

```sql
SELECT
FROM
JOIN
WHERE
GROUP BY
HAVING
ORDER BY
LIMIT
```

But logically it is processed roughly like this:

1. `FROM`
2. `JOIN`
3. `WHERE`
4. `GROUP BY`
5. `HAVING`
6. `SELECT`
7. `ORDER BY`
8. `LIMIT`


In [ ]:
display(query(conn,"""select * from departments;"""))
display(query(conn,"""select * from students;"""))
display(query(conn,"""select * from instructors;"""))

display(query(conn,"""select * from courses;"""))
display(query(conn,"""select * from enrollments;"""))
display(query(conn,"""select * from payments;"""))

,department_id,department_name
0,1,Data Science
1,2,Software Engineering
2,3,Business Analytics
3,4,Cloud Computing


,student_id,student_name,email,city,registration_date
0,1,Rahul Kumar,rahul.kumar@example.com,Patna,2026-01-05
1,2,Priya Singh,priya.singh@example.com,Kolkata,2026-01-06
2,3,Amit Raj,amit.raj@example.com,Delhi,2026-01-07
3,4,Sneha Verma,sneha.verma@example.com,Patna,2026-01-10
4,5,Aditya Sharma,aditya.sharma@example.com,Mumbai,2026-01-12
5,6,Neha Gupta,neha.gupta@example.com,Ranchi,2026-01-13
6,7,Rohan Das,rohan.das@example.com,Pune,2026-01-15
7,8,Farhan Ali,farhan.ali@example.com,Patna,2026-01-18
8,9,Mansi Roy,mansi.roy@example.com,Kolkata,2026-01-20
9,10,Kiran Nair,kiran.nair@example.com,Bengaluru,2026-01-22


,instructor_id,instructor_name,email,department_id
0,1,Dr. Meera Iyer,meera.iyer@example.com,1
1,2,Arjun Sen,arjun.sen@example.com,2
2,3,Kavita Rao,kavita.rao@example.com,3
3,4,Ravi Menon,ravi.menon@example.com,4


,course_id,course_name,department_id,instructor_id,fee,level
0,101,Python for Beginners,2,2,4999.0,Beginner
1,102,SQL and RDBMS Masterclass,1,1,6999.0,Beginner
2,103,Machine Learning Basics,1,1,11999.0,Intermediate
3,104,Business Dashboarding,3,3,8999.0,Intermediate
4,105,Advanced Data Engineering,1,1,15999.0,Advanced
5,106,Cloud Database Fundamentals,4,4,10999.0,Beginner
6,107,Data Visualization with Python,3,3,7999.0,Beginner
7,108,MLOps Fundamentals,1,1,13999.0,Advanced


,enrollment_id,student_id,course_id,enrollment_date,status
0,1001,1,101,2026-02-01,Active
1,1002,1,102,2026-02-03,Completed
2,1003,2,102,2026-02-04,Active
3,1004,3,103,2026-02-05,Active
4,1005,4,104,2026-02-07,Cancelled
5,1006,5,105,2026-02-08,Active
6,1007,6,101,2026-02-08,Completed
7,1008,2,104,2026-02-09,Active
8,1009,7,106,2026-02-10,Active
9,1010,8,107,2026-02-11,Completed


,payment_id,enrollment_id,amount,payment_date,payment_status
0,501,1001,4999.0,2026-02-01,Paid
1,502,1002,6999.0,2026-02-03,Paid
2,503,1003,6999.0,2026-02-04,Pending
3,504,1004,11999.0,2026-02-05,Paid
4,505,1005,0.0,2026-02-07,Refunded
5,506,1006,15999.0,2026-02-08,Paid
6,507,1007,4999.0,2026-02-08,Paid
7,508,1008,8999.0,2026-02-09,Pending
8,509,1009,10999.0,2026-02-10,Paid
9,510,1010,7999.0,2026-02-11,Paid


In [ ]:
query(conn,"""
  select
    e.enrollment_id,
    s.student_name,
    c.course_name,
    e.enrollment_date
  from enrollments e
  left join students s
  on e.student_id = s.student_id
  left join courses c
  on c.course_id = e.course_id
  order by e.enrollment_id;
""")

,enrollment_id,student_name,course_name,enrollment_date
0,1001,Rahul Kumar,Python for Beginners,2026-02-01
1,1002,Rahul Kumar,SQL and RDBMS Masterclass,2026-02-03
2,1003,Priya Singh,SQL and RDBMS Masterclass,2026-02-04
3,1004,Amit Raj,Machine Learning Basics,2026-02-05
4,1005,Sneha Verma,Business Dashboarding,2026-02-07
5,1006,Aditya Sharma,Advanced Data Engineering,2026-02-08
6,1007,Neha Gupta,Python for Beginners,2026-02-08
7,1008,Priya Singh,Business Dashboarding,2026-02-09
8,1009,Rohan Das,Cloud Database Fundamentals,2026-02-10
9,1010,Farhan Ali,Data Visualization with Python,2026-02-11


## 4. INNER JOIN

Returns only matching records from both tables.


In [ ]:
query(conn, """
SELECT
    e.enrollment_id,
    s.student_name,
    c.course_name,
    e.enrollment_date,
    e.status
FROM enrollments e
INNER JOIN students s
    ON e.student_id = s.student_id
INNER JOIN courses c
    ON e.course_id = c.course_id
ORDER BY e.enrollment_id;
""")


,enrollment_id,student_name,course_name,enrollment_date,status
0,1001,Rahul Kumar,Python for Beginners,2026-02-01,Active
1,1002,Rahul Kumar,SQL and RDBMS Masterclass,2026-02-03,Completed
2,1003,Priya Singh,SQL and RDBMS Masterclass,2026-02-04,Active
3,1004,Amit Raj,Machine Learning Basics,2026-02-05,Active
4,1005,Sneha Verma,Business Dashboarding,2026-02-07,Cancelled
5,1006,Aditya Sharma,Advanced Data Engineering,2026-02-08,Active
6,1007,Neha Gupta,Python for Beginners,2026-02-08,Completed
7,1008,Priya Singh,Business Dashboarding,2026-02-09,Active
8,1009,Rohan Das,Cloud Database Fundamentals,2026-02-10,Active
9,1010,Farhan Ali,Data Visualization with Python,2026-02-11,Completed


## 5. LEFT JOIN

Returns all records from the left table and matched records from the right table.

If no matching record exists, right-side columns become `NULL`.


In [ ]:
query(conn, """
SELECT
    s.student_id,
    s.student_name,
    c.course_name,
    e.status
FROM students s
LEFT JOIN enrollments e
    ON s.student_id = e.student_id
LEFT JOIN courses c
    ON e.course_id = c.course_id
ORDER BY s.student_id;
""")


,student_id,student_name,course_name,status
0,1,Rahul Kumar,Python for Beginners,Active
1,1,Rahul Kumar,SQL and RDBMS Masterclass,Completed
2,2,Priya Singh,SQL and RDBMS Masterclass,Active
3,2,Priya Singh,Business Dashboarding,Active
4,3,Amit Raj,SQL and RDBMS Masterclass,Completed
5,3,Amit Raj,Machine Learning Basics,Active
6,4,Sneha Verma,Business Dashboarding,Cancelled
7,5,Aditya Sharma,Advanced Data Engineering,Active
8,6,Neha Gupta,Python for Beginners,Completed
9,7,Rohan Das,Cloud Database Fundamentals,Active


## 6. Find Missing Records with LEFT JOIN

Find students who have not enrolled in any course.


In [ ]:
query(conn, """
SELECT
    s.student_id,
    s.student_name,
    s.city
FROM students s
LEFT JOIN enrollments e
    ON s.student_id = e.student_id
WHERE e.enrollment_id IS NULL;
""")


,student_id,student_name,city


In [ ]:
query(conn,"""
select
  s.student_name,
  s.student_id,
  e.enrollment_id,
  e.status
from students s
left join enrollments e
  on s.student_id = e.student_id
order by e.enrollment_id;
""")

,student_name,student_id,enrollment_id,status
0,Rahul Kumar,1,1001,Active
1,Rahul Kumar,1,1002,Completed
2,Priya Singh,2,1003,Active
3,Amit Raj,3,1004,Active
4,Sneha Verma,4,1005,Cancelled
5,Aditya Sharma,5,1006,Active
6,Neha Gupta,6,1007,Completed
7,Priya Singh,2,1008,Active
8,Rohan Das,7,1009,Active
9,Farhan Ali,8,1010,Completed


## 7. All Courses Even Without Enrollment

This is a common business query.


In [ ]:
query(conn, """
SELECT
    c.course_id,
    c.course_name,
    e.enrollment_id,
    e.status
FROM courses c
LEFT JOIN enrollments e
    ON c.course_id = e.course_id
ORDER BY c.course_id;
""")


,course_id,course_name,enrollment_id,status
0,101,Python for Beginners,1001,Active
1,101,Python for Beginners,1013,Active
2,101,Python for Beginners,1007,Completed
3,102,SQL and RDBMS Masterclass,1003,Active
4,102,SQL and RDBMS Masterclass,1011,Active
5,102,SQL and RDBMS Masterclass,1002,Completed
6,102,SQL and RDBMS Masterclass,1014,Completed
7,103,Machine Learning Basics,1004,Active
8,104,Business Dashboarding,1008,Active
9,104,Business Dashboarding,1005,Cancelled


## 8. CROSS JOIN

Returns all possible combinations.

Use carefully because it can create a very large result set.


In [ ]:
query(conn, """
SELECT
    s.student_name,
    c.course_name
FROM students s
CROSS JOIN courses c
LIMIT 20;
""")


,student_name,course_name
0,Rahul Kumar,Python for Beginners
1,Rahul Kumar,SQL and RDBMS Masterclass
2,Rahul Kumar,Machine Learning Basics
3,Rahul Kumar,Business Dashboarding
4,Rahul Kumar,Advanced Data Engineering
5,Rahul Kumar,Cloud Database Fundamentals
6,Rahul Kumar,Data Visualization with Python
7,Rahul Kumar,MLOps Fundamentals
8,Priya Singh,Python for Beginners
9,Priya Singh,SQL and RDBMS Masterclass


## 9. SELF JOIN

A self join joins a table with itself.

Example: find pairs of students from the same city.


In [ ]:
query(conn, """
SELECT
    s1.student_name AS student_1,
    s1.student_id,
    s2.student_name AS student_2,
    s2.student_id,
    s1.city
FROM students s1
INNER JOIN students s2
    ON s1.city = s2.city
   AND s1.student_id > s2.student_id
ORDER BY s1.city;
""")


,student_1,student_id,student_2,student_id,city
0,Mansi Roy,9,Priya Singh,2,Kolkata
1,Sneha Verma,4,Rahul Kumar,1,Patna
2,Farhan Ali,8,Rahul Kumar,1,Patna
3,Farhan Ali,8,Sneha Verma,4,Patna


In [ ]:
query(conn, """
SELECT
    s1.student_name AS student_1,
    s2.student_name AS student_2,
    s1.city
FROM students s1
INNER JOIN students s2
    ON s1.city = s2.city
ORDER BY s1.city;
""")


,student_1,student_2,city
0,Kiran Nair,Kiran Nair,Bengaluru
1,Amit Raj,Amit Raj,Delhi
2,Priya Singh,Mansi Roy,Kolkata
3,Priya Singh,Priya Singh,Kolkata
4,Mansi Roy,Mansi Roy,Kolkata
5,Mansi Roy,Priya Singh,Kolkata
6,Aditya Sharma,Aditya Sharma,Mumbai
7,Rahul Kumar,Farhan Ali,Patna
8,Rahul Kumar,Rahul Kumar,Patna
9,Rahul Kumar,Sneha Verma,Patna


## 10. Aggregate Functions

| Function | Purpose |
|---|---|
| COUNT | Count rows |
| SUM | Add numbers |
| AVG | Average |
| MIN | Minimum |
| MAX | Maximum |


In [ ]:
query(conn, """
SELECT
    COUNT(*) AS total_students
FROM students;
""")


,total_students
0,10


In [ ]:
query(conn, """
SELECT
    SUM(amount) AS total_paid_revenue
FROM payments
WHERE payment_status = 'Paid';
""")


,total_paid_revenue
0,89990.0


## 11. GROUP BY

`GROUP BY` creates groups and then applies aggregate functions.


In [ ]:
query(conn, """
SELECT
    city,
    COUNT(*) AS total_students
FROM students
GROUP BY city
ORDER BY total_students DESC;
""")


## 12. GROUP BY with JOIN

Count enrollments per course.


In [ ]:
query(conn, """
SELECT
    c.course_name,
    COUNT(e.enrollment_id) AS total_enrollments
FROM courses c
LEFT JOIN enrollments e
    ON c.course_id = e.course_id
GROUP BY c.course_id, c.course_name
ORDER BY total_enrollments DESC;
""")


## 13. HAVING

`WHERE` filters rows before grouping.

`HAVING` filters grouped results.


In [ ]:
query(conn, """
SELECT
    c.course_name,
    COUNT(e.enrollment_id) AS total_enrollments
FROM courses c
INNER JOIN enrollments e
    ON c.course_id = e.course_id
GROUP BY c.course_id, c.course_name
HAVING COUNT(e.enrollment_id) > 1
ORDER BY total_enrollments DESC;
""")


## 14. CASE Statement

`CASE` adds conditional logic in SQL.


In [ ]:
query(conn, """
SELECT
    course_name,
    fee,
    CASE
        WHEN fee >= 12000 THEN 'Premium'
        WHEN fee BETWEEN 7000 AND 11999 THEN 'Standard'
        ELSE 'Affordable'
    END AS fee_category
FROM courses
ORDER BY fee DESC;
""")


## 15. Revenue by Course

This query uses joins, aggregation and `CASE`.


In [ ]:
query(conn, """
SELECT
    c.course_name,
    COUNT(e.enrollment_id) AS total_enrollments,
    SUM(CASE WHEN p.payment_status = 'Paid' THEN p.amount ELSE 0 END) AS paid_revenue,
    SUM(CASE WHEN p.payment_status = 'Pending' THEN p.amount ELSE 0 END) AS pending_amount
FROM courses c
LEFT JOIN enrollments e
    ON c.course_id = e.course_id
LEFT JOIN payments p
    ON e.enrollment_id = p.enrollment_id
GROUP BY c.course_id, c.course_name
ORDER BY paid_revenue DESC;
""")


## 16. Subquery

A subquery is a query inside another query.

Find courses with fee higher than average fee.


In [ ]:
query(conn, """
SELECT
    course_name,
    fee
FROM courses
WHERE fee > (
    SELECT AVG(fee)
    FROM courses
)
ORDER BY fee DESC;
""")


## 17. Subquery with IN

Find students enrolled in SQL and RDBMS Masterclass.


In [ ]:
query(conn, """
SELECT
    student_id,
    student_name,
    city
FROM students
WHERE student_id IN (
    SELECT student_id
    FROM enrollments
    WHERE course_id = (
        SELECT course_id
        FROM courses
        WHERE course_name = 'SQL and RDBMS Masterclass'
    )
);
""")


,student_id,student_name,city
0,1,Rahul Kumar,Patna
1,2,Priya Singh,Kolkata
2,3,Amit Raj,Delhi
3,9,Mansi Roy,Kolkata


In [ ]:
query(conn,"""
  select
  c.course_name,
  count(e.enrollment_id) as total_Enrollments,
  sum(case when p.payment_status = "Paid" then p.amount else 0 end) as paid_revenue,
  sum(case when p.payment_status = "Pending" then p.amount else 0 end) as pending_amount
from courses c
left join enrollments e
on c.course_id = e.course_id
left join payments p
on e.enrollment_id = p.enrollment_id
group by c.course_id, c.course_name
order by paid_revenue desc;
""")

,course_name,total_Enrollments,paid_revenue,pending_amount
0,Advanced Data Engineering,1,15999.0,0.0
1,Python for Beginners,3,14997.0,0.0
2,MLOps Fundamentals,1,13999.0,0.0
3,SQL and RDBMS Masterclass,4,13998.0,13998.0
4,Machine Learning Basics,1,11999.0,0.0
5,Cloud Database Fundamentals,1,10999.0,0.0
6,Data Visualization with Python,1,7999.0,0.0
7,Business Dashboarding,2,0.0,8999.0


In [ ]:
query(conn, """
create view vw_student_payment_summary as
select
  s.student_id,
  s.student_name,
  s.city,

  count(distinct e.enrollment_id) as total_enrollments,

  coalesce(sum(
      case
        when p.payment_status = "Paid" then p.amount
        else 0
      end
  ),0) as total_paid_revenue,


  coalesce(sum(
      case
        when p.payment_status = "Pending" then p.amount
        else 0
      end
  ),0) as total_pending_amount,

  case
  when coalesce(sum(
      case
        when p.payment_status = "Pending" then p.amount
        else 0
      end
  ),0) >=20000 then 'Premium Student'

  when coalesce(sum(
      case
        when p.payment_status = "Pending" then p.amount
        else 0
      end
  ),0) >=10000 then 'Standard Student'

  else 'Basic Student'
  end as student_category

from students s
left join enrollments e
on s.student_id = e.student_id
left join payments p
on e.enrollment_id = p.enrollment_id
group by s.student_id, s.student_name, s.city
""")



DatabaseError: Execution failed on sql '
create view vw_student_payment_summary as
select
  s.student_id,
  s.student_name,
  s.city,

  count(distinct e.enrollment_id) as total_enrollments,

  coalesce(sum(
      case
        when p.payment_status = "Paid" then p.amount
        else 0
      end
  ),0) as total_paid_revenue,


  coalesce(sum(
      case
        when p.payment_status = "Pending" then p.amount
        else 0
      end
  ),0) as total_pending_amount,

  case
  when coalesce(sum(
      case
        when p.payment_status = "Pending" then p.amount
        else 0
      end
  ),0) >=20000 then 'Premium Student'

  when coalesce(sum(
      case
        when p.payment_status = "Pending" then p.amount
        else 0
      end
  ),0) >=10000 then 'Standard Student'

  else 'Basic Student'
  end as student_category

from students s
left join enrollments e
on s.student_id = e.student_id
left join payments p
on e.enrollment_id = p.enrollment_id
group by s.student_id, s.student_name, 

In [ ]:
query(conn, "select * from vw_student_payment_summary;")

,student_id,student_name,city,total_enrollments,total_paid_revenue,total_pending_amount,student_category
0,1,Rahul Kumar,Patna,2,11998.0,0.0,Basic Student
1,2,Priya Singh,Kolkata,2,0.0,15998.0,Standard Student
2,3,Amit Raj,Delhi,2,18998.0,0.0,Basic Student
3,4,Sneha Verma,Patna,1,0.0,0.0,Basic Student
4,5,Aditya Sharma,Mumbai,1,15999.0,0.0,Basic Student
5,6,Neha Gupta,Ranchi,1,4999.0,0.0,Basic Student
6,7,Rohan Das,Pune,1,10999.0,0.0,Basic Student
7,8,Farhan Ali,Patna,2,12998.0,0.0,Basic Student
8,9,Mansi Roy,Kolkata,1,0.0,6999.0,Basic Student
9,10,Kiran Nair,Bengaluru,1,13999.0,0.0,Basic Student


## 18. Correlated Subquery

A correlated subquery depends on the outer query.


In [ ]:
query(conn, """
SELECT
    c.course_name,
    c.fee,
    d.department_name
FROM courses c
INNER JOIN departments d
    ON c.department_id = d.department_id
WHERE c.fee > (
    SELECT AVG(c2.fee)
    FROM courses c2
    WHERE c2.department_id = c.department_id
)
ORDER BY d.department_name, c.fee DESC;
""")


## 19. Common Table Expression

CTE makes SQL easier to read.


In [ ]:
query(conn, """
WITH course_revenue AS (
    SELECT
        c.course_id,
        c.course_name,
        SUM(CASE WHEN p.payment_status = 'Paid' THEN p.amount ELSE 0 END) AS paid_revenue
    FROM courses c
    LEFT JOIN enrollments e
        ON c.course_id = e.course_id
    LEFT JOIN payments p
        ON e.enrollment_id = p.enrollment_id
    GROUP BY c.course_id, c.course_name
)
SELECT *
FROM course_revenue
WHERE paid_revenue >= 10000
ORDER BY paid_revenue DESC;
""")


## 20. Multiple CTEs

Department-wise revenue and enrollment count.


In [ ]:
query(conn, """
WITH course_metrics AS (
    SELECT
        c.course_id,
        c.department_id,
        COUNT(e.enrollment_id) AS enrollments,
        SUM(CASE WHEN p.payment_status = 'Paid' THEN p.amount ELSE 0 END) AS paid_revenue
    FROM courses c
    LEFT JOIN enrollments e
        ON c.course_id = e.course_id
    LEFT JOIN payments p
        ON e.enrollment_id = p.enrollment_id
    GROUP BY c.course_id, c.department_id
),
department_metrics AS (
    SELECT
        d.department_name,
        SUM(cm.enrollments) AS total_enrollments,
        SUM(cm.paid_revenue) AS total_paid_revenue
    FROM departments d
    LEFT JOIN course_metrics cm
        ON d.department_id = cm.department_id
    GROUP BY d.department_name
)
SELECT *
FROM department_metrics
ORDER BY total_paid_revenue DESC;
""")


## 21. Window Functions

Window functions calculate values across related rows without collapsing rows.

Examples:
- `RANK()`
- `ROW_NUMBER()`
- `SUM() OVER`
- `AVG() OVER`


In [ ]:
query(conn, """
SELECT
    course_name,
    fee,
    RANK() OVER (ORDER BY fee DESC) AS fee_rank
FROM courses
ORDER BY fee_rank;
""")


## 22. Window Function with PARTITION BY

Rank courses inside each department.


In [ ]:
query(conn, """
SELECT
    d.department_name,
    c.course_name,
    c.fee,
    RANK() OVER (
        PARTITION BY d.department_name
        ORDER BY c.fee DESC
    ) AS rank_in_department
FROM courses c
INNER JOIN departments d
    ON c.department_id = d.department_id
ORDER BY d.department_name, rank_in_department;
""")


## 23. Running Total


In [ ]:
query(conn, """
SELECT
    payment_date,
    payment_id,
    amount,
    SUM(amount) OVER (
        ORDER BY payment_date, payment_id
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS running_paid_revenue
FROM payments
WHERE payment_status = 'Paid'
ORDER BY payment_date, payment_id;
""")


## 24. Views

A view is a saved SQL query.

It behaves like a virtual table.


In [ ]:
exec_script(conn, """
DROP VIEW IF EXISTS vw_enrollment_details;

CREATE VIEW vw_enrollment_details AS
SELECT
    e.enrollment_id,
    s.student_name,
    s.city,
    c.course_name,
    d.department_name,
    i.instructor_name,
    e.enrollment_date,
    e.status,
    p.amount,
    p.payment_status
FROM enrollments e
INNER JOIN students s
    ON e.student_id = s.student_id
INNER JOIN courses c
    ON e.course_id = c.course_id
INNER JOIN departments d
    ON c.department_id = d.department_id
INNER JOIN instructors i
    ON c.instructor_id = i.instructor_id
LEFT JOIN payments p
    ON e.enrollment_id = p.enrollment_id;
""")

query(conn, "SELECT * FROM vw_enrollment_details ORDER BY enrollment_id;")


## 25. Indexes

Indexes can speed up searching and joins.

They should be used carefully because they also add storage and write overhead.


In [ ]:
exec_script(conn, """
CREATE INDEX IF NOT EXISTS idx_students_city ON students(city);
CREATE INDEX IF NOT EXISTS idx_enrollments_student_course ON enrollments(student_id, course_id);
""")

query(conn, """
SELECT name, tbl_name, sql
FROM sqlite_master
WHERE type = 'index' AND sql IS NOT NULL;
""")


## 26. EXPLAIN QUERY PLAN


In [ ]:
query(conn, """
EXPLAIN QUERY PLAN
SELECT *
FROM students
WHERE city = 'Patna';
""")


## 27. Transactions

A transaction groups multiple database changes.

If all operations succeed, use `COMMIT`.

If any operation fails, use `ROLLBACK`.


In [ ]:
try:
    conn.execute("BEGIN;")
    conn.execute("""
    INSERT INTO enrollments (enrollment_id, student_id, course_id, enrollment_date, status)
    VALUES (?, ?, ?, ?, ?);
    """, (2001, 10, 101, "2026-03-01", "Active"))

    conn.execute("""
    INSERT INTO payments (payment_id, enrollment_id, amount, payment_date, payment_status)
    VALUES (?, ?, ?, ?, ?);
    """, (9001, 2001, 4999, "2026-03-01", "Paid"))

    conn.commit()
    print("Transaction committed.")
except Exception as e:
    conn.rollback()
    print("Transaction rolled back:", e)

query(conn, "SELECT * FROM enrollments WHERE enrollment_id = 2001;")


## 28. Practice Query Set

Write SQL for:

1. Top 3 courses by paid revenue.
2. Cities with more than one student.
3. Students who completed at least one course.
4. Courses with zero paid revenue.
5. Rank departments by paid revenue.
6. Average fee per department.
7. Students with pending payments.
8. Instructors whose courses generated more than 10000 paid revenue.
9. Each student's total paid amount.
10. Most popular course by enrollment count.


In [ ]:
# Practice 1: Top 3 courses by paid revenue
query(conn, """
SELECT
    course_name,
    SUM(CASE WHEN payment_status = 'Paid' THEN amount ELSE 0 END) AS paid_revenue
FROM vw_enrollment_details
GROUP BY course_name
ORDER BY paid_revenue DESC
LIMIT 3;
""")


In [ ]:
# Practice 2: Students with pending payments
query(conn, """
SELECT
    student_name,
    course_name,
    amount,
    payment_status
FROM vw_enrollment_details
WHERE payment_status = 'Pending'
ORDER BY amount DESC;
""")


In [ ]:
# Practice 3: Rank departments by revenue
query(conn, """
WITH dept_revenue AS (
    SELECT
        department_name,
        SUM(CASE WHEN payment_status = 'Paid' THEN amount ELSE 0 END) AS paid_revenue
    FROM vw_enrollment_details
    GROUP BY department_name
)
SELECT
    department_name,
    paid_revenue,
    RANK() OVER (ORDER BY paid_revenue DESC) AS revenue_rank
FROM dept_revenue;
""")


## 29. Mini Assessment

Answer:

1. Difference between `INNER JOIN` and `LEFT JOIN`.
2. Difference between `WHERE` and `HAVING`.
3. Why is `GROUP BY` used?
4. Why do we use CTE?
5. How is a window function different from aggregation?
6. Why are transactions important?


## 30. Notebook 2 Summary

You learned:

- Joins
- Aggregations
- `GROUP BY` and `HAVING`
- Subqueries
- CTEs
- CASE
- Window functions
- Views
- Indexes
- Transactions

Next notebook: build a complete Python + RDBMS mini project.
